# Dashboard Qualité de Service — Réseau Infrabel

**Auteur** : Tahar Guenfoud
**Données** : [Open Data Infrabel](https://opendata.infrabel.be)

---

Ce notebook analyse la ponctualité et la fiabilité du réseau ferroviaire belge à partir des données publiques d'Infrabel.
L'idée de départ : reproduire le type d'analyse qu'un Data Analyst ferait pour préparer le reporting trimestriel du Contrat de Performance.

## Plan

| Étape | Ce qu'on fait |
|---|---|
| **1. Extract** | Téléchargement des 5 datasets Open Data Infrabel |
| **2. Transform** | Nettoyage, renommage des colonnes, parsing des dates |
| **3. EDA** | Exploration des données — tendances, anomalies, questions métier |
| **4. KPIs** | Calcul des indicateurs : Ponctualité, Fiabilité, Trains en retard |
| **5. Visualisation** | Graphiques finaux pour le dashboard |

---
## 0. Imports & Configuration

In [1]:
import requests
import pandas as pd
import os
import time

# Dossier de sauvegarde (données partagées DataScience)
RAW_DIR   = "../data/raw"
CLEAN_DIR = "../data/clean"

print("✅ Imports OK")

✅ Imports OK


---
## Étape 1 — Extract

On télécharge les 5 datasets directement depuis l'API Open Data Infrabel.
Pas de fichiers locaux — les données sont toujours à jour à chaque exécution.

In [2]:
BASE = (
    "https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets"
    "/{}/exports/csv?lang=fr&timezone=Europe%2FBrussels&use_labels=true&delimiter=%3B"
)

DATASETS = {
    "ponctualite_par_gare"    : "maandelijkse-stiptheid-per-stopplaats",
    "causes_retards"          : "oorzaken-vertraging-per-maand",
    "ponctualite_par_moment"  : "nationale-stiptheid-per-moment-en-per-maand",
    "trains_supprimes"        : "afgeschafte-treinen-per-maand-vanaf-2020",
    "kpi_contrat_performance" : "indicatoren-performantie-contract",
}

dfs = {}
for name, dataset_id in DATASETS.items():
    dfs[name] = pd.read_csv(BASE.format(dataset_id), sep=";")
    print(f"✅ {name:<35} {dfs[name].shape[0]:>6,} lignes × {dfs[name].shape[1]} cols")

✅ ponctualite_par_gare                27,902 lignes × 13 cols
✅ causes_retards                         430 lignes × 14 cols
✅ ponctualite_par_moment                 488 lignes × 9 cols
✅ trains_supprimes                        74 lignes × 7 cols
✅ kpi_contrat_performance                177 lignes × 13 cols


---
## Étape 2 — Transform

Chaque dataset brut a ses propres problèmes : noms de colonnes en néerlandais, dates mal typées, colonnes redondantes.
On nettoie tout ici avant de toucher à l'analyse.

### 2.1 — Ponctualité par Gare

Le dataset principal : ponctualité mensuelle pour chaque gare du réseau.

In [4]:
# Explorer les colonnes brutes
df = dfs["ponctualite_par_gare"]
df.head(10)

,Date,Point d'arrêt,Point d'arrêt.1,Point d'arrêt.2,ID point opérationnel,Classification,Classification.1,Classification.2,Ponctualité,Nombre total de trains,Nombre de trains ponctuels,Geo Point,Geo Shape
0,2022-06,BORDET,BORDET,BORDET,191,Stopplaats,Point d'arrêt,Stopping point,90.829694,5038.0,4576.0,"50.877774256061116, 4.410014404677236","{""coordinates"": [4.410014404677236, 50.8777742..."
1,2022-05,BORNEM,BORNEM,BORNEM,192,Stopplaats,Point d'arrêt,Stopping point,93.168232,1171.0,1091.0,"51.098985014071545, 4.240696491019956","{""coordinates"": [4.240696491019956, 51.0989850..."
2,2022-02,BOUSSU,BOUSSU,BOUSSU,195,Station,Gare,Station,89.281768,905.0,808.0,"50.43603991330167, 3.795971741039548","{""coordinates"": [3.795971741039548, 50.4360399..."
3,2022-03,BOUSSU,BOUSSU,BOUSSU,195,Station,Gare,Station,88.568486,971.0,860.0,"50.43603991330167, 3.795971741039548","{""coordinates"": [3.795971741039548, 50.4360399..."
4,2022-06,BOUSSU,BOUSSU,BOUSSU,195,Station,Gare,Station,86.830835,934.0,811.0,"50.43603991330167, 3.795971741039548","{""coordinates"": [3.795971741039548, 50.4360399..."
5,2022-05,BOUWEL,BOUWEL,BOUWEL,199,Stopplaats,Point d'arrêt,Stopping point,86.973948,998.0,868.0,"51.165843512547234, 4.74758182017099","{""coordinates"": [4.74758182017099, 51.16584351..."
6,2022-06,BOUWEL,BOUWEL,BOUWEL,199,Stopplaats,Point d'arrêt,Stopping point,84.300000,1000.0,843.0,"51.165843512547234, 4.74758182017099","{""coordinates"": [4.74758182017099, 51.16584351..."
7,2022-07,BOUWEL,BOUWEL,BOUWEL,199,Stopplaats,Point d'arrêt,Stopping point,91.633146,1243.0,1139.0,"51.165843512547234, 4.74758182017099","{""coordinates"": [4.74758182017099, 51.16584351..."
8,2022-03,BRACQUEGNIES,BRACQUEGNIES,BRACQUEGNIES,201,Station,Gare,Station,94.778661,881.0,835.0,"50.474362121919384, 4.126350513837885","{""coordinates"": [4.126350513837885, 50.4743621..."
9,2022-07,BRACQUEGNIES,BRACQUEGNIES,BRACQUEGNIES,201,Station,Gare,Station,96.577017,818.0,790.0,"50.474362121919384, 4.126350513837885","{""coordinates"": [4.126350513837885, 50.4743621..."


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27902 entries, 0 to 27901
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Date                        27902 non-null  object 
 1   Point d'arrêt               27902 non-null  object 
 2   Point d'arrêt.1             27902 non-null  object 
 3   Point d'arrêt.2             27902 non-null  object 
 4   ID point opérationnel       27902 non-null  int64  
 5   Classification              27902 non-null  object 
 6   Classification.1            27902 non-null  object 
 7   Classification.2            27902 non-null  object 
 8   Ponctualité                 27902 non-null  float64
 9   Nombre total de trains      27902 non-null  float64
 10  Nombre de trains ponctuels  27902 non-null  float64
 11  Geo Point                   27866 non-null  object 
 12  Geo Shape                   27866 non-null  object 
dtypes: float64(3), int64(1), object

In [6]:
df.isnull().sum()

Date                           0
Point d'arrêt                  0
Point d'arrêt.1                0
Point d'arrêt.2                0
ID point opérationnel          0
Classification                 0
Classification.1               0
Classification.2               0
Ponctualité                    0
Nombre total de trains         0
Nombre de trains ponctuels     0
Geo Point                     36
Geo Shape                     36
dtype: int64

In [7]:
df["Date"].unique()

array(['2022-06', '2022-05', '2022-02', '2022-03', '2022-07', '2022-01',
       '2022-04', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10',
       '2023-11', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03',
       '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01',
       '2023-02', '2023-03', '2023-04', '2023-05', '2025-04', '2025-05',
       '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11',
       '2023-12', '2025-12', '2024-01', '2026-01', '2024-02', '2026-02',
       '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08',
       '2024-09', '2024-10'], dtype=object)

In [8]:
df_gare = dfs["ponctualite_par_gare"].copy()

# Renommer (13 colonnes dans l'ordre exact)
df_gare.columns = [
    "date",
    "nom_gare_fr",
    "nom_gare_nl",
    "nom_gare_de",
    "id_gare",
    "classification_fr",
    "classification_nl",
    "classification_de",
    "ponctualite_pct",
    "nb_trains",
    "nb_trains_ponctuels",
    "geo_point",
    "geo_shape"
]

# Parser la date
df_gare["date"] = pd.to_datetime(df_gare["date"], format="%Y-%m")

# Vérification
print(f"Shape : {df_gare.shape}")
print(f"Valeurs manquantes :\n{df_gare.isnull().sum()}")
df_gare.head(3)

Shape : (27902, 13)
Valeurs manquantes :
date                    0
nom_gare_fr             0
nom_gare_nl             0
nom_gare_de             0
id_gare                 0
classification_fr       0
classification_nl       0
classification_de       0
ponctualite_pct         0
nb_trains               0
nb_trains_ponctuels     0
geo_point              36
geo_shape              36
dtype: int64


,date,nom_gare_fr,nom_gare_nl,nom_gare_de,id_gare,classification_fr,classification_nl,classification_de,ponctualite_pct,nb_trains,nb_trains_ponctuels,geo_point,geo_shape
0,2022-06-01,BORDET,BORDET,BORDET,191,Stopplaats,Point d'arrêt,Stopping point,90.829694,5038.0,4576.0,"50.877774256061116, 4.410014404677236","{""coordinates"": [4.410014404677236, 50.8777742..."
1,2022-05-01,BORNEM,BORNEM,BORNEM,192,Stopplaats,Point d'arrêt,Stopping point,93.168232,1171.0,1091.0,"51.098985014071545, 4.240696491019956","{""coordinates"": [4.240696491019956, 51.0989850..."
2,2022-02-01,BOUSSU,BOUSSU,BOUSSU,195,Station,Gare,Station,89.281768,905.0,808.0,"50.43603991330167, 3.795971741039548","{""coordinates"": [3.795971741039548, 50.4360399..."


In [9]:
# Voir quelques exemples côte à côte
df_gare[["nom_gare_fr", "nom_gare_nl", "nom_gare_de"]].drop_duplicates().head(20)

,nom_gare_fr,nom_gare_nl,nom_gare_de
0,BORDET,BORDET,BORDET
1,BORNEM,BORNEM,BORNEM
2,BOUSSU,BOUSSU,BOUSSU
5,BOUWEL,BOUWEL,BOUWEL
8,BRACQUEGNIES,BRACQUEGNIES,BRACQUEGNIES
10,BRAINE-L'ALLEUD,BRAINE-L'ALLEUD,BRAINE-L'ALLEUD
11,BRAINE-LE-COMTE,BRAINE-LE-COMTE,BRAINE-LE-COMTE
14,BRESSOUX,BRESSOUX,BRESSOUX
15,BRUGELETTE,BRUGELETTE,BRUGELETTE
18,BRUGGE,BRUGGE,BRUGGE


In [10]:
# Combien de fois les 3 colonnes sont identiques ?
identiques = (df_gare["nom_gare_fr"] == df_gare["nom_gare_nl"]).sum()
print(f"FR = NL : {identiques} fois sur {len(df_gare)}")

identiques2 = (df_gare["nom_gare_fr"] == df_gare["nom_gare_de"]).sum()
print(f"FR = DE : {identiques2} fois sur {len(df_gare)}")

FR = NL : 26579 fois sur 27902
FR = DE : 26579 fois sur 27902


In [11]:
# Supprimer les colonnes inutiles
df_gare = df_gare.drop(columns=["nom_gare_nl", "nom_gare_de",
                                 "classification_nl", "classification_de",
                                 "geo_shape"])

print(f"Colonnes restantes : {df_gare.columns.tolist()}")
print(f"Shape : {df_gare.shape}")

Colonnes restantes : ['date', 'nom_gare_fr', 'id_gare', 'classification_fr', 'ponctualite_pct', 'nb_trains', 'nb_trains_ponctuels', 'geo_point']
Shape : (27902, 8)


In [12]:
display(df_gare.head())
df_gare.info()

,date,nom_gare_fr,id_gare,classification_fr,ponctualite_pct,nb_trains,nb_trains_ponctuels,geo_point
0,2022-06-01,BORDET,191,Stopplaats,90.829694,5038.0,4576.0,"50.877774256061116, 4.410014404677236"
1,2022-05-01,BORNEM,192,Stopplaats,93.168232,1171.0,1091.0,"51.098985014071545, 4.240696491019956"
2,2022-02-01,BOUSSU,195,Station,89.281768,905.0,808.0,"50.43603991330167, 3.795971741039548"
3,2022-03-01,BOUSSU,195,Station,88.568486,971.0,860.0,"50.43603991330167, 3.795971741039548"
4,2022-06-01,BOUSSU,195,Station,86.830835,934.0,811.0,"50.43603991330167, 3.795971741039548"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27902 entries, 0 to 27901
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 27902 non-null  datetime64[ns]
 1   nom_gare_fr          27902 non-null  object        
 2   id_gare              27902 non-null  int64         
 3   classification_fr    27902 non-null  object        
 4   ponctualite_pct      27902 non-null  float64       
 5   nb_trains            27902 non-null  float64       
 6   nb_trains_ponctuels  27902 non-null  float64       
 7   geo_point            27866 non-null  object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 1.7+ MB


### 2.2 — Causes des Retards

Ce dataset est particulièrement intéressant : il ventile les retards par **responsable** (Infrabel, SNCB, Tiers).
C'est ce qui permet de savoir *à cause de qui* les trains sont en retard.

| Colonne | Ce qu'elle contient |
|---|---|
| `responsable` | Infrabel / SNCB / Tiers / Robustesse systémique |
| `nb_trains_en_retard` | Trains en retard attribués à ce responsable ce mois |
| `perte_ponctualite` | Minutes de retard totales causées ce mois |
| `proportion_pct` | Part (%) de ce responsable dans les retards du mois |
| `*_ytd` | Cumul depuis le 1er janvier (Year-To-Date) |

> **YTD** : les colonnes `_ytd` permettent de suivre la tendance annuelle sans être perturbé par les variations d'un mois à l'autre.

In [13]:
# Explorer les colonnes brutes
df = dfs["causes_retards"]
display(df.head(3))
df.info()

,Année,Année/mois,Mois,Responsable NL,Responsable,Responsable EN,Nombre de repérages de trains en retard,Nombre total de repérages de trains,Perte de ponctualité,% proportion,Nombre de repérages de trains en retard YTD,Nombre total de repérages de trains YTD,Perte de ponctualité YTD,% proportion YTD
0,2026,2026-02,2,NMBS,SNCB,SNCB/NMBS,2845.051982,108663,2.62,38.80,6644.055909,216529,3.07,42.76
1,2026,2026-02,2,Andere,Autres,Others,357.277337,108663,0.33,4.87,627.669317,216529,0.29,4.04
2,2026,2026-02,2,Infrabel,Infrabel,Infrabel,1192.658477,108663,1.10,16.26,2108.672943,216529,0.97,13.57


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 430 entries, 0 to 429
Data columns (total 14 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Année                                        430 non-null    int64  
 1   Année/mois                                   430 non-null    object 
 2   Mois                                         430 non-null    int64  
 3   Responsable NL                               430 non-null    object 
 4   Responsable                                  430 non-null    object 
 5   Responsable EN                               430 non-null    object 
 6   Nombre de repérages de trains en retard      430 non-null    float64
 7   Nombre total de repérages de trains          430 non-null    int64  
 8   Perte de ponctualité                         430 non-null    float64
 9   % proportion                                 430 non-null    float64
 10  No

In [14]:
df_causes = dfs["causes_retards"].copy()

print(f"Shape : {df_causes.shape}")
print(f"Valeurs manquantes :\n{df_causes.isnull().sum()}")
df_causes.head(3)

Shape : (430, 14)
Valeurs manquantes :
Année                                          0
Année/mois                                     0
Mois                                           0
Responsable NL                                 0
Responsable                                    0
Responsable EN                                 0
Nombre de repérages de trains en retard        0
Nombre total de repérages de trains            0
Perte de ponctualité                           0
% proportion                                   0
Nombre de repérages de trains en retard YTD    0
Nombre total de repérages de trains YTD        0
Perte de ponctualité YTD                       0
% proportion YTD                               0
dtype: int64


,Année,Année/mois,Mois,Responsable NL,Responsable,Responsable EN,Nombre de repérages de trains en retard,Nombre total de repérages de trains,Perte de ponctualité,% proportion,Nombre de repérages de trains en retard YTD,Nombre total de repérages de trains YTD,Perte de ponctualité YTD,% proportion YTD
0,2026,2026-02,2,NMBS,SNCB,SNCB/NMBS,2845.051982,108663,2.62,38.80,6644.055909,216529,3.07,42.76
1,2026,2026-02,2,Andere,Autres,Others,357.277337,108663,0.33,4.87,627.669317,216529,0.29,4.04
2,2026,2026-02,2,Infrabel,Infrabel,Infrabel,1192.658477,108663,1.10,16.26,2108.672943,216529,0.97,13.57


In [15]:
# Valeurs uniques dans chaque colonne
print("FR :", df["Responsable"].unique())
print("NL :", df["Responsable NL"].unique())
print("EN :", df["Responsable EN"].unique())

FR : ['SNCB' 'Autres' 'Infrabel' 'Robustesse systémique' 'Tiers']
NL : ['NMBS' 'Andere' 'Infrabel' 'Robuustheid van het systeem' 'Derden']
EN : ['SNCB/NMBS' 'Others' 'Infrabel' 'Systemic robustness' 'Third parties']


In [16]:
# Vérifier si FR et NL sont toujours différents
identiques = (df["Responsable"] == df["Responsable NL"]).sum()
print(f"FR = NL : {identiques} fois sur {len(df)}")

identiques2 = (df["Responsable"] == df["Responsable EN"]).sum()
print(f"FR = EN : {identiques2} fois sur {len(df)}")

FR = NL : 86 fois sur 430
FR = EN : 86 fois sur 430


In [17]:
df_causes.columns = [
    "annee", "date", "mois",
    "responsable_nl", "responsable", "responsable_en",
    "nb_trains_en_retard", "nb_trains_total",
    "perte_ponctualite", "proportion_pct",
    "nb_trains_en_retard_ytd", "nb_trains_total_ytd",
    "perte_ponctualite_ytd", "proportion_ytd_pct"
]

# Parser la date
df_causes["date"] = pd.to_datetime(df_causes["date"], format="%Y-%m")

# Supprimer les colonnes redondantes
df_causes = df_causes.drop(columns=["annee", "mois", "responsable_nl", "responsable_en"])

print(f"Shape : {df_causes.shape}")
print(f"\nResponsables : {df_causes['responsable'].unique()}")
df_causes.head()

Shape : (430, 10)

Responsables : ['SNCB' 'Autres' 'Infrabel' 'Robustesse systémique' 'Tiers']


,date,responsable,nb_trains_en_retard,nb_trains_total,perte_ponctualite,proportion_pct,nb_trains_en_retard_ytd,nb_trains_total_ytd,perte_ponctualite_ytd,proportion_ytd_pct
0,2026-02-01,SNCB,2845.051982,108663,2.62,38.80,6644.055909,216529,3.07,42.76
1,2026-02-01,Autres,357.277337,108663,0.33,4.87,627.669317,216529,0.29,4.04
2,2026-02-01,Infrabel,1192.658477,108663,1.10,16.26,2108.672943,216529,0.97,13.57
3,2026-02-01,Robustesse systémique,1352.323513,108663,1.24,18.44,2791.595867,216529,1.29,17.97
4,2026-02-01,Tiers,1585.688690,108663,1.46,21.62,3365.005965,216529,1.55,21.66


In [18]:
df_causes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 430 entries, 0 to 429
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   date                     430 non-null    datetime64[ns]
 1   responsable              430 non-null    object        
 2   nb_trains_en_retard      430 non-null    float64       
 3   nb_trains_total          430 non-null    int64         
 4   perte_ponctualite        430 non-null    float64       
 5   proportion_pct           430 non-null    float64       
 6   nb_trains_en_retard_ytd  430 non-null    float64       
 7   nb_trains_total_ytd      430 non-null    int64         
 8   perte_ponctualite_ytd    430 non-null    float64       
 9   proportion_ytd_pct       430 non-null    float64       
dtypes: datetime64[ns](1), float64(6), int64(2), object(1)
memory usage: 33.7+ KB


### 2.3 — Ponctualité par Moment

Même chose que la ponctualité par gare, mais segmentée par période de la journée : Matin, Journée, Soir, Weekend.

In [19]:
df = dfs["ponctualite_par_moment"]
display(df.head())
df.info()

,Mois,Instant,Instant.1,Instant.2,Ponctualité,Nombre de trains,Trains ayant moins de 6 min. de retard,Minutes de retard,Année
0,2016-02,Ochtendspits,Pointe du matin,Morning peak hour,86.902427,16316,14179,43540,2016
1,2016-03,Daluren,Heures creuses,Off-peak hours,91.098429,50991,46452,100860,2016
2,2016-04,Ochtendspits,Pointe du matin,Morning peak hour,89.015654,15076,13420,34654,2016
3,2016-05,Avondspits,Pointe du soir,Evening peak hour,83.002208,13590,11280,45655,2016
4,2016-06,Ochtendspits,Pointe du matin,Morning peak hour,86.849442,15064,13083,42389,2016


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 488 entries, 0 to 487
Data columns (total 9 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Mois                                    488 non-null    object 
 1   Instant                                 488 non-null    object 
 2   Instant.1                               488 non-null    object 
 3   Instant.2                               488 non-null    object 
 4   Ponctualité                             488 non-null    float64
 5   Nombre de trains                        488 non-null    int64  
 6   Trains ayant moins de 6 min. de retard  488 non-null    int64  
 7   Minutes de retard                       488 non-null    int64  
 8   Année                                   488 non-null    int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 34.4+ KB


In [20]:
print(df["Instant"].unique())
print(df["Instant.1"].unique())
print(df["Instant.2"].unique())

['Ochtendspits' 'Daluren' 'Avondspits' 'Weekends']
['Pointe du matin' 'Heures creuses' 'Pointe du soir' 'Weekends']
['Morning peak hour' 'Off-peak hours' 'Evening peak hour' 'Weekends']


In [ ]:
df_moment = dfs["ponctualite_par_moment"].copy()

df_moment.columns = [
    "date",
    "periode_nl",
    "periode",
    "periode_en",
    "ponctualite_pct",
    "nb_trains",
    "nb_trains_ponctuels",
    "nb_minutes_retard",
    "annee"
]

# Parser la date
df_moment["date"] = pd.to_datetime(df_moment["date"], format="%Y-%m")

# Supprimer les colonnes redondantes
df_moment = df_moment.drop(columns=["periode_nl", "periode_en", "annee"])

print(f"Shape : {df_moment.shape}")
print(f"\nPériodes disponibles : {df_moment['periode'].unique()}")
df_moment.head()

In [ ]:
df_moment.info()

### 2.4 — Trains Supprimés

Les trains annulés — l'indicateur de **fiabilité** du réseau.
Différent de la ponctualité : un train supprimé n'est pas en retard, il n'existe tout simplement pas.

In [ ]:
df = dfs["trains_supprimes"]
display(df.head(3))
df.info()

In [ ]:
df_suppression = dfs["trains_supprimes"].copy()

df_suppression.columns = [
    "date",
    "nb_trains_supprimes_total",
    "nb_trains_supprimes_partiel",
    "nb_trains_supprimes_entier",
    "nb_trains",
    "pct_trains_supprimes",
    "annee"
]

# Parser la date
df_suppression["date"] = pd.to_datetime(df_suppression["date"], format="%Y-%m")

# Supprimer colonne redondante
df_suppression = df_suppression.drop(columns=["annee"])

print(f"Shape : {df_suppression.shape}")
df_suppression.head()

In [ ]:
df_suppression.info()

### 2.5 — KPIs Contrat de Performance

Les indicateurs officiels du Contrat de Performance 2023-2032 entre l'État belge et Infrabel.
Utile pour calibrer nos KPIs maison sur les vrais objectifs contractuels.

In [ ]:
df = dfs["kpi_contrat_performance"]
display(df.head(3))
df.info()

In [ ]:
df.isna().sum()

In [ ]:
# Quelles catégories existent ?
print("Catégories :")
print(df["Catégorie"].value_counts())

print("\nTypes :")
print(df["Type"].value_counts())

print("\nSous-catégories :")
print(df["Sous-catégorie"].value_counts())

> **Choix de nettoyage :** j'ai supprimé les colonnes avec trop de valeurs manquantes et sans intérêt pour le dashboard.
> J'ai gardé `objectif` et `valeur_reelle` malgré leurs lacunes — ce sont les deux colonnes centrales pour comparer performance réelle vs objectif contractuel.

In [ ]:
df_kpi = dfs["kpi_contrat_performance"].copy()

# Renommer
df_kpi.columns = [
    "annee", "id_indicateur", "type_indicateur",
    "categorie_nl", "categorie",
    "sous_categorie_nl", "sous_categorie",
    "objectif", "valeur_reelle", "bonus",
    "remediation", "seuil_superieur", "unite"
]

# Garder seulement Ponctualité + Fiabilité
df_kpi = df_kpi[df_kpi["categorie"].isin(["Ponctualité", "Fiabilité & Vitesse Commerciale"])]

# Supprimer colonnes inutiles
df_kpi = df_kpi.drop(columns=[
    "categorie_nl", "sous_categorie_nl",
    "seuil_superieur",
    "bonus", "remediation"
])

print(f"Shape : {df_kpi.shape}")
print(f"\nIndicateurs restants :")
print(df_kpi["sous_categorie"].unique())
df_kpi.head()

In [ ]:
df_kpi.info()

---
## Étape 3 — EDA (Exploratory Data Analysis)

Avant de construire le dashboard, on explore les données pour comprendre ce qu'elles racontent.
Chaque section répond à une question métier concrète.

| Section | Question |
|---|---|
| **3.1** | La ponctualité s'améliore-t-elle sur les dernières années ? |
| **3.2** | Quelles gares concentrent le plus de retards ? |
| **3.3** | Les heures de pointe sont-elles vraiment les pires ? |
| **3.4** | Qui est responsable des retards : Infrabel, SNCB ou des tiers ? |
| **3.5** | Comment évolue la fiabilité — les trains sont-ils de moins en moins annulés ? |

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio                                                       
pio.renderers.default = "notebook"



In [ ]:
df_gare["date"].unique()

### 3.1 — Tendance Temporelle

Est-ce que la ponctualité s'améliore ou empire ? On commence par ça — c'est la question la plus importante.

In [ ]:
# Agréger par mois (moyenne nationale)
df_tendance = df_gare.groupby("date")["ponctualite_pct"].mean().reset_index()
display(df_tendance.head())
# Graphique
fig = px.line(
    df_tendance,
    x="date",
    y="ponctualite_pct",
    title="📈 Tendance de la Ponctualité Nationale (2022-2025)",
    labels={"date": "Mois", "ponctualite_pct": "Ponctualité (%)"},
)

# Ligne seuil objectif contractuel (90%)
fig.add_hline(
    y=90,
    line_dash="dash",
    line_color="red",
    annotation_text="Objectif 90%"
)

fig.show()

In [ ]:
# Zoomer sur la période août-novembre 2023
masque = (df_causes["date"] >= "2023-08-01") & (df_causes["date"] <= "2023-11-30")
df_crise = df_causes[masque]

# Qui est responsable ?
print(df_crise.groupby("responsable")["perte_ponctualite"].sum().sort_values(ascending=False))

In [ ]:
# Moyenne annuelle de ponctualité
df_annuel = df_gare.groupby(df_gare["date"].dt.year)["ponctualite_pct"].mean().reset_index()
df_annuel.columns = ["annee", "ponctualite_moyenne"]
df_annuel["ponctualite_moyenne"] = df_annuel["ponctualite_moyenne"].round(2)
print(df_annuel)

In [ ]:
# Filtrer depuis janvier 2024
df_2024_plus = df_tendance[df_tendance["date"] >= "2024-01-01"].copy()

# Calculer la pente (régression linéaire simple)
from numpy.polynomial import polynomial as P
import numpy as np

x = np.arange(len(df_2024_plus))
y = df_2024_plus["ponctualite_pct"].values
coefs = P.polyfit(x, y, 1)

print(f"Pente mensuelle : +{coefs[1]:.3f}% par mois")
print(f"Soit +{coefs[1]*12:.2f}% par an")


### 3.2 — Top 10 Gares Problématiques

Toutes les gares ne se valent pas. On cherche celles qui concentrent le plus de retards.

In [ ]:
df_gare.head()

In [ ]:
# Calculer le nombre de trains en retard par ligne
df_gare['nb_trains_retard'] = df_gare['nb_trains'] - df_gare['nb_trains_ponctuels']

# Grouper par gare pour avoir le cumul total de trains en retard
top_10_volume = df_gare.groupby("nom_gare_fr")['nb_trains_retard'].sum().sort_values(ascending=False).head(10).reset_index()
display(top_10_volume)

In [ ]:
# 1. Calculer le volume de retards et la ponctualité moyenne par gare
df_analyse_gares = df_gare.groupby("nom_gare_fr").agg({
    'nb_trains_retard': 'sum',
    'ponctualite_pct': 'mean'
}).reset_index()

# 2. Prendre le Top 10 par volume de retards
top_10_final = df_analyse_gares.sort_values(by="nb_trains_retard", ascending=False).head(10)

# 3. Arrondir pour la lisibilité
top_10_final['ponctualite_pct'] = top_10_final['ponctualite_pct'].round(2)

In [ ]:
import plotly.express as px

fig = px.bar(
    top_10_final,
    x="nb_trains_retard",
    y="nom_gare_fr",
    orientation='h',
    color="ponctualite_pct", # La couleur montre la performance
    color_continuous_scale="RdYlGn", # Rouge (mauvais) -> Jaune -> Vert (bon)
    title="🚆 Top 10 des Gares par Volume de Retards (Impact Passager)",
    labels={
        "nb_trains_retard": "Nombre Total de Trains en Retard",
        "nom_gare_fr": "Gare",
        "ponctualite_pct": "Ponctualité Moyenne (%)"
    },
    text="nb_trains_retard" # Affiche le chiffre exact au bout de la barre
)

# Amélioration du design
fig.update_traces(textposition='outside')
fig.update_layout(
    yaxis={'categoryorder':'total ascending'}, # Met la pire gare (Brussel-Zuid) en haut
    plot_bgcolor='white',
    xaxis_title="Volume de retards (Cumulé)",
    height=600
)

fig.show()

### 3.3 — Analyse par Moment de la Journée

Le matin est-il vraiment le pire moment ? Et le weekend, est-ce mieux ?

In [ ]:
display(df_moment.head())
df_moment.info()

In [ ]:
df_moment["periode"].unique()

In [ ]:
display(df_analyse_gares.head())
df_analyse_gares.info()

In [ ]:
df_moment["df_moment_retard"] = df_moment["nb_trains"] - df_moment["nb_trains_ponctuels"]
periode = df_moment.groupby("periode")['df_moment_retard'].sum().sort_values(ascending=False).head(10).reset_index()
display(periode)

In [ ]:
df_moment.groupby("periode")["ponctualite_pct"].mean().sort_values().round(2)

In [ ]:
df_par_periode = df_moment.groupby("periode").agg(
    nb_trains_retard=("df_moment_retard", "sum"),
    ponctualite_pct=("ponctualite_pct", "mean")
).reset_index().round(2)

display(df_par_periode.sort_values("ponctualite_pct"))

In [ ]:
fig = px.bar(
    df_par_periode.sort_values("ponctualite_pct"),
    x="periode",
    y="ponctualite_pct",
    color="ponctualite_pct",
    color_continuous_scale="RdYlGn",
    text="ponctualite_pct",
    title="⏰ Ponctualité par Période (Matin / Soir / Creuses / Weekend)",
    labels={"ponctualite_pct": "Ponctualité (%)", "periode": "Période"}
)
fig.add_hline(y=90, line_dash="dash", line_color="red", annotation_text="Objectif 90%")
fig.show()

### 3.4 — Causes des Retards

La question clé : qui est responsable ? Infrabel (infrastructure), SNCB (matériel/personnel), ou des tiers (vandales, accidents) ?

In [ ]:
display(df_causes.head())
df_causes.info()

In [ ]:
# masque = (df_causes["date"] >= "2023-08-01") & (df_causes["date"] <= "2023-11-30")
# df_crise = df_causes[masque]

# Qui est responsable ?
print(df_causes.groupby("responsable")["perte_ponctualite"].sum().sort_values(ascending=False))

### 3.5 — Trains Supprimés / Fiabilité

Un réseau ponctuel mais avec beaucoup d'annulations, ce n'est pas un bon réseau.
On regarde ici la fiabilité : est-ce que les trains circulent vraiment ?

In [ ]:
display(df_suppression.head())
df_suppression.info()

In [ ]:
# 3.5 — Fiabilité du réseau (trains non supprimés)
df_suppression["Mensuel brut"] = (100 - df_suppression["pct_trains_supprimes"]).round(2)
df_suppression["Tendance (moy. 3 mois)"] = df_suppression["Mensuel brut"].rolling(3, center=True).mean().round(2)

moy = df_suppression["Mensuel brut"].mean().round(2)
print(f"Moyenne: {moy}% | Min: {df_suppression['Mensuel brut'].min()}% | Max: {df_suppression['Mensuel brut'].max()}%")

fig = px.line(
    df_suppression, x="date", y=["Mensuel brut", "Tendance (moy. 3 mois)"],
    title="🚫 Évolution de la Fiabilité — % Trains Non Annulés",
    labels={"date": "Mois", "value": "Fiabilité (%)", "variable": ""},
    color_discrete_map={"Mensuel brut": "rgba(99,155,255,0.3)", "Tendance (moy. 3 mois)": "#3b82f6"}
)
fig.add_hline(y=99, line_dash="dash", line_color="red", annotation_text="Objectif 99%")
fig.add_hline(y=moy, line_dash="dot", line_color="orange", annotation_text=f"Moyenne {moy}%")
fig.update_layout(yaxis=dict(range=[93, 100]), plot_bgcolor="white", hovermode="x unified")
fig.show()


---
## Étape 4 — KPIs Métier

On calcule les 3 indicateurs du Contrat de Performance Infrabel :
- **Ponctualité** : % de trains arrivés avec moins de 6 min de retard (objectif : 90%)
- **Fiabilité** : % de trains non annulés (objectif : 99%)
- **Trains en retard** : volume total sur l'année (impact économique)

In [ ]:
# ================================
# ÉTAPE 4 — KPIs Métier Infrabel
# ================================

# Dernière année disponible
derniere_annee = df_gare["date"].dt.year.max()
df_annee = df_gare[df_gare["date"].dt.year == derniere_annee]

# KPI 1 — Ponctualité
kpi_ponctualite = df_annee["ponctualite_pct"].mean().round(2)

# KPI 2 — Fiabilité
kpi_fiabilite = (100 - df_suppression["pct_trains_supprimes"].mean()).round(2)

# KPI 3 — Minutes perdues
kpi_trains_retard = int(df_annee["nb_trains_retard"].sum())

print(f"📅 Année : {derniere_annee}")
print(f"⏱️  Ponctualité  : {kpi_ponctualite}%  (objectif : 90%)")
print(f"✅  Fiabilité    : {kpi_fiabilite}%  (objectif : 99%)")
print(f"🚆  Trains retard : {kpi_trains_retard:,}")


---
## Étape 5 — Visualisations

Les graphiques finaux du dashboard. Chaque visu répond à une question précise :

| # | Graphique | Question |
|---|---|---|
| 5.1 | KPI Cards | On est à combien ? |
| 5.2 | Line Chart | Ça s'améliore ? |
| 5.3 | Bar Chart | Quelles gares sont les pires ? |
| 5.4 | Donut Chart | À cause de qui ? |
| 5.5 | Heatmap | Quand est-ce que ça déraille ? |

In [ ]:
# ============================================================
# ÉTAPE 5.1 — KPI Cards (Simple & Pro)
# ============================================================

kpis = [
    {"label": "⏱️ Ponctualité",    "value": kpi_ponctualite,   "ref": 90,  "suffix": "%"},
    {"label": "✅ Fiabilité",       "value": kpi_fiabilite,     "ref": 99,  "suffix": "%"},
    {"label": "🚆 Trains en retard","value": kpi_trains_retard, "ref": None,"suffix": ""},
]

fig_kpi = go.Figure()

for i, k in enumerate(kpis):
    fig_kpi.add_trace(go.Indicator(
        mode="number+delta" if k["ref"] else "number",
        value=k["value"],
        number={"suffix": k["suffix"], "valueformat": ",.0f" if not k["suffix"] else ".1f",
                "font": {"size": 48, "color": "#ef4444" if not k["ref"] else "#1e293b"}},
        delta={"reference": k["ref"], "suffix": k["suffix"],
               "increasing": {"color": "#22c55e"}, "decreasing": {"color": "#ef4444"}} if k["ref"] else {},
        title={"text": f"{k['label']}<br><span style='font-size:.7em;color:#94a3b8'>"
                       f"{'Objectif : ' + str(k['ref']) + k['suffix'] if k['ref'] else 'Total ' + str(derniere_annee)}</span>"},
        domain={"x": [i/3, (i+1)/3], "y": [0, 1]}
    ))

fig_kpi.update_layout(
    title={"text": f"📊 KPIs Réseau Infrabel — {derniere_annee}", "x": 0.5, "xanchor": "center"},
    paper_bgcolor="#f8fafc", height=280,
    margin={"t": 70, "b": 10, "l": 10, "r": 10}
)

fig_kpi.show()

In [ ]:
# ============================================================
# ÉTAPE 5.2 — Line Chart : Tendance Ponctualité
# ============================================================

df_tendance = df_annee.groupby("date")["ponctualite_pct"].mean().round(2).reset_index()
df_tendance["tendance_3m"] = df_tendance["ponctualite_pct"].rolling(3, center=True).mean().round(2)

fig_line = px.line(
    df_tendance, x="date", y=["ponctualite_pct", "tendance_3m"],
    title=f"📈 Évolution de la Ponctualité — {derniere_annee}",
    labels={"value": "Ponctualité (%)", "date": "Mois", "variable": ""},
    color_discrete_map={"ponctualite_pct": "rgba(59,130,246,0.4)", "tendance_3m": "#3b82f6"},
    markers=True, height=400
)

fig_line.add_hline(y=90, line_dash="dash", line_color="red", annotation_text="Objectif 90%")
fig_line.update_layout(yaxis=dict(range=[80, 100], ticksuffix="%"),
                       paper_bgcolor="#f8fafc", plot_bgcolor="white",
                       hovermode="x unified", legend=dict(orientation="h", y=-0.2))
fig_line.show()

In [ ]:
# ============================================================
# ÉTAPE 5.3 — Bar Chart : Top 5 Gares les plus problématiques
# ================================================

top5 = (
    df_annee.assign(nb_trains_retard=df_annee["nb_trains"] - df_annee["nb_trains_ponctuels"])
    .groupby("nom_gare_fr")["nb_trains_retard"].sum()
    .sort_values(ascending=False).head(5).reset_index()
)

fig_bar = px.bar(
    top5, x="nb_trains_retard", y="nom_gare_fr", orientation="h",
    title=f"🚉 Top 5 Gares — Trains en Retard ({derniere_annee})",
    labels={"nb_trains_retard": "Trains en retard", "nom_gare_fr": ""},
    color="nb_trains_retard", color_continuous_scale="RdYlGn_r",
    text="nb_trains_retard", height=350
)

fig_bar.update_traces(texttemplate="%{text:,}", textposition="outside")
fig_bar.update_layout(paper_bgcolor="#f8fafc", plot_bgcolor="white",
                      coloraxis_showscale=False, yaxis=dict(autorange="reversed"))
fig_bar.show()

In [ ]:
# ============================================================
# ÉTAPE 5.4 — Donut Chart : Responsables des Retards
# ============================================================

fig = px.pie(
    df_causes.groupby("responsable")["perte_ponctualite"].sum().reset_index(),
    values="perte_ponctualite", names="responsable",
    title="🔍 Répartition des Responsables de Retards",
    color_discrete_sequence=["#ef4444", "#3b82f6", "#f59e0b"],
    hole=0.4
)
fig.update_traces(textinfo="percent+label", pull=[0.05, 0, 0])
fig.update_layout(paper_bgcolor="#f8fafc", showlegend=False, height=380)
fig.show()

In [ ]:
# ============================================================
# ÉTAPE 5.5 — Heatmap : Ponctualité par Mois × Période
# ============================================================

df_heat = df_moment.copy()
df_heat["mois"] = df_heat["date"].dt.strftime("%b %Y")   # ex: "Jan 2024"

pivot = df_heat.pivot_table(
    index="periode", columns="mois",
    values="ponctualite_pct", aggfunc="mean"
).round(1)

fig_heat = px.imshow(
    pivot,
    title="🌡️ Ponctualité par Période & Mois (%)",
    color_continuous_scale="RdYlGn",
    zmin=80, zmax=100,
    text_auto=True,
    aspect="auto", height=380
)

fig_heat.update_layout(
    paper_bgcolor="#f8fafc",
    coloraxis_colorbar=dict(title="%", ticksuffix="%"),
    xaxis_title="", yaxis_title=""
)
fig_heat.show()